# Scientific Paper RAG System

A comprehensive RAG pipeline for scientific papers using:
- **LlamaIndex** for orchestration
- **Weaviate** for multimodal vector storage
- **BAAI/bge-m3** for embeddings
- **Gemini 2.5 Pro** for generation
- **Gemini 2.0 Flash** for evaluation
- **Hybrid Fusion Retriever** with Reciprocal Rank Fusion
- **Arize Phoenix** for observability

## 1. Install Dependencies

In [32]:
# # Run this cell first to install all required packages
# !pip install -q \
#     llama-index \
#     llama-index-llms-gemini \
#     llama-index-embeddings-huggingface \
#     llama-index-vector-stores-weaviate \
#     llama-index-retrievers-bm25 \
#     weaviate-client \
#     openinference-instrumentation-llama-index \
#     ragas \
#     pymupdf \
#     pillow \
#     pdf2image \
#     pytesseract \
#     easyocr \
#     tabula-py \
#     pystemmer \
#     python-dotenv \
#     nest-asyncio \
#     sentence-transformers \
#     FlagEmbedding

## 2. Imports and Setup

In [33]:
import os
import io
import re
import asyncio
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import pdfplumber

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
import fitz  # PyMuPDF
from PIL import Image
from dotenv import load_dotenv
import Stemmer
import tabula 

# LlamaIndex imports
from llama_index.core import (
    Document,
    VectorStoreIndex,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

# LLM and Embeddings
from llama_index.llms.gemini import Gemini
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from google import genai
from google.genai import types

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


## 3. Configuration

In [45]:
load_dotenv()

@dataclass
class Config:
    # API Keys
    GOOGLE_API_KEY: str = os.getenv("GOOGLE_API_KEY", "")
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    
    LLM_MODEL: str = "models/gemini-2.5-flash"
    JUDGE_LLM_MODEL: str = "gpt-4o-mini"
    EMBEDDING_MODEL: str = "BAAI/bge-m3"
    
    # Chunking
    CHUNK_SIZE: int = 512
    CHUNK_OVERLAP: int = 128

    TEXT_CHUNK_SIZE = 512
    TEXT_CHUNK_OVERLAP = 128

    SEC_CHUNK_SIZE = 256
    SEC_CHUNK_OVERLAP = 50

    TABLE_CHUNK_SIZE = 1000
    TABLE_CHUNK_OVERLAP = 128

    FIGURE_CHUNK_SIZE = 500
    FIGURE_CHUNK_OVERLAP = 120

    REF_CHUNK_SIZE = 700
    REF_CHUNK_OVERLAP = 120
    
    # Retrieval
    TOP_K: int = 5
    VECTOR_WEIGHT: float = 0.8
    BM25_WEIGHT: float = 0.2
    
    # PDF Path
    PDF_PATH: str = "2502.12110v11.pdf"

config = Config()

if not config.GOOGLE_API_KEY:
    print("⚠️ GOOGLE_API_KEY not found!")
    print("Please set it: export GOOGLE_API_KEY='your-key-here'")
else:
    print(f"✅ API Key loaded: {config.GOOGLE_API_KEY[:10]}...")

if not config.OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found!")
    print("Get one at: https://platform.openai.com/api-keys")
else:
    print(f"✅ OpenAI API Key: {config.OPENAI_API_KEY[:10]}...")

✅ API Key loaded: AIzaSyBWSx...
✅ OpenAI API Key: sk-proj-_f...


## 4. Initialize Phoenix Tracing (Optional)

In [46]:
# Uncomment to enable Phoenix tracing
# import phoenix as px
# from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
# from phoenix.otel import register

# px.launch_app()
# tracer_provider = register(project_name="scientific-paper-rag")
# LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)
# print("✅ Phoenix tracing at: http://localhost:6006")

## 5. Initialize LLMs and Embeddings

In [47]:
# Initialize Gemini LLMs
llm = Gemini(
    model=config.LLM_MODEL,
    api_key=config.GOOGLE_API_KEY,
    temperature=0.1,
    max_tokens=1024,
    context_window=8192,
)

judge_llm = OpenAI(
    api_key=config.OPENAI_API_KEY,
    model=config.JUDGE_LLM_MODEL,
    max_tokens=1024,
    temperature=0.1,
)

# Initialize BGE-M3 Embedding Model
embed_model = HuggingFaceEmbedding(
    model_name=config.EMBEDDING_MODEL,
    trust_remote_code=True,
    embed_batch_size=8,
)

# Set global settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = config.CHUNK_SIZE
Settings.chunk_overlap = config.CHUNK_OVERLAP

print(f"✅ LLM: {config.LLM_MODEL}")
print(f"✅ Judge: {config.JUDGE_LLM_MODEL}")
print(f"✅ Embeddings: {config.EMBEDDING_MODEL}")

✅ LLM: models/gemini-2.5-flash
✅ Judge: gpt-4o-mini
✅ Embeddings: BAAI/bge-m3


## 6. Initialize Weaviate

In [48]:
# No vector store initialization needed - using in-memory
print("✅ Using LlamaIndex in-memory vector store")

✅ Using LlamaIndex in-memory vector store


## 7. Document Processing Functions

In [49]:
# =============================================================================
# SECTION 7: DOCUMENT PROCESSING FUNCTIONS (ENHANCED)
# =============================================================================

def clean_text(text: str) -> str:
    """Clean extracted text from PDFs."""
    text = re.sub(r'\x00', '', text) 
    text = re.sub(r'\f', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\\s+\\d+\\s+$', '', text)
    text = re.sub(r'\b(nan|NaN)\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\\[\\d+(?:,\\s*\\d+)*\\]', lambda m: m.group(0), text)
    return text.strip()


# -----------------------------------------------------------------------------
# STRUCTURE ANALYSIS FUNCTIONS
# -----------------------------------------------------------------------------

def analyze_document_structure(pdf_path: str) -> Dict[str, Any]:
    """
    Analyze PDF structure to extract:
    - Title
    - Authors
    - Emails
    - Organizations/Affiliations
    - Abstract
    - Section headings
    """
    print("📋 Analyzing document structure...")
    
    structure = {
        "title": "",
        "authors": [],
        "emails": [],
        "organizations": [],
        "abstract": "",
        "sections": [],
        "raw_first_pages": ""
    }
    
    doc = fitz.open(pdf_path)
    
    # Extract text from first 2-3 pages for header analysis
    header_text = ""
    for page_num in range(min(3, len(doc))):
        page = doc[page_num]
        header_text += page.get_text("text") + "\n"
    
    structure["raw_first_pages"] = header_text
    
    # Extract title (usually the largest font on first page, or first major text)
    structure["title"] = extract_title(doc, header_text)
    
    # Extract authors
    structure["authors"] = extract_authors(header_text)
    
    # Extract emails
    structure["emails"] = extract_emails(header_text)
    
    # Extract organizations/affiliations
    structure["organizations"] = extract_organizations(header_text)
    
    # Extract abstract
    structure["abstract"] = extract_abstract(header_text)
    
    # Extract section headings from entire document
    full_text = ""
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text += page.get_text("text") + "\n"
    
    structure["sections"] = extract_section_headings(full_text)
    
    doc.close()
    
    # Print summary
    print(f"   📌 Title: {structure['title'][:60]}..." if len(structure['title']) > 60 else f"   📌 Title: {structure['title']}")
    print(f"   👥 Authors: {len(structure['authors'])} found")
    print(f"   📧 Emails: {len(structure['emails'])} found")
    print(f"   🏢 Organizations: {len(structure['organizations'])} found")
    print(f"   📝 Abstract: {'Found' if structure['abstract'] else 'Not found'}")
    print(f"   📑 Sections: {len(structure['sections'])} found")
    
    return structure


def extract_title(doc, header_text: str) -> str:
    """Extract paper title using font analysis and heuristics."""
    # Try to get title from first page using font size analysis
    first_page = doc[0]
    blocks = first_page.get_text("dict")["blocks"]
    
    title_candidates = []
    
    for block in blocks:
        if "lines" in block:
            for line in block["lines"]:
                for span in line["spans"]:
                    font_size = span["size"]
                    text = span["text"].strip()
                    
                    # Title is usually large font (>14pt) and near top
                    if font_size > 14 and len(text) > 10:
                        title_candidates.append((font_size, text))
    
    # Sort by font size (largest first)
    title_candidates.sort(key=lambda x: x[0], reverse=True)
    
    if title_candidates:
        # Combine consecutive large font texts that might be multi-line title
        title = title_candidates[0][1]
        return clean_text(title)
    
    # Fallback: Use first substantial line
    lines = header_text.split('\n')
    for line in lines[:10]:
        line = line.strip()
        if len(line) > 20 and not re.match(r'^(Abstract|Introduction|\d+\.)', line):
            return clean_text(line)
    
    return ""


def extract_authors(header_text: str) -> List[str]:
    """Extract author names from header text."""
    authors = []
    
    # Common patterns for author names
    # Pattern 1: Names with superscripts (e.g., "John Doe¹")
    # Pattern 2: Names separated by commas or "and"
    # Pattern 3: Names before email addresses
    
    lines = header_text.split('\n')
    
    # Look for author block (usually after title, before abstract)
    in_author_section = False
    author_lines = []
    
    for i, line in enumerate(lines[:30]):  # Check first 30 lines
        line = line.strip()
        
        # Skip empty lines and common non-author content
        if not line:
            continue
        if re.match(r'^(Abstract|Introduction|\d+\.|Keywords|arXiv)', line, re.IGNORECASE):
            in_author_section = False
            continue
        
        # Look for lines with multiple capitalized words (potential names)
        # Authors often have format: "First Last, First Last, ..."
        if re.search(r'[A-Z][a-z]+\s+[A-Z][a-z]+', line):
            # Check if this looks like names (not a sentence)
            words = line.split()
            capitalized_ratio = sum(1 for w in words if w[0].isupper()) / len(words) if words else 0
            
            if capitalized_ratio > 0.5 and len(line) < 200:
                # Extract individual names
                # Remove affiliations markers (superscripts, asterisks)
                cleaned = re.sub(r'[∗†‡§¶\d,]+', ' ', line)
                cleaned = re.sub(r'\s+', ' ', cleaned).strip()
                
                # Split by common separators
                potential_names = re.split(r'\s+and\s+|,\s*', cleaned)
                
                for name in potential_names:
                    name = name.strip()
                    # Validate it looks like a name (2-4 words, starts with capital)
                    name_parts = name.split()
                    if 2 <= len(name_parts) <= 4 and all(p[0].isupper() for p in name_parts if p):
                        if name not in authors:
                            authors.append(name)
    
    return authors[:20]  # Limit to 20 authors


def extract_emails(header_text: str) -> List[str]:
    """Extract email addresses from text."""
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    emails = re.findall(email_pattern, header_text)
    return list(set(emails))


def extract_organizations(header_text: str) -> List[str]:
    """Extract organizations/affiliations from header text."""
    organizations = []
    
    # Common organization keywords
    org_keywords = [
        r'University', r'Institute', r'Laboratory', r'Lab',
        r'Department', r'School', r'College', r'Center',
        r'Corporation', r'Inc\.', r'Ltd\.', r'Research',
        r'Google', r'Microsoft', r'Meta', r'OpenAI', r'DeepMind',
        r'Facebook', r'Amazon', r'IBM', r'NVIDIA'
    ]
    
    pattern = r'(' + '|'.join(org_keywords) + r')[^,\n]*'
    
    matches = re.findall(pattern, header_text, re.IGNORECASE)
    
    # Also look for lines that seem like affiliations
    lines = header_text.split('\n')
    for line in lines[:40]:
        line = line.strip()
        for keyword in org_keywords:
            if re.search(keyword, line, re.IGNORECASE):
                # Clean the line
                cleaned = re.sub(r'[∗†‡§¶\d]+', '', line).strip()
                if len(cleaned) > 10 and cleaned not in organizations:
                    organizations.append(cleaned)
                break
    
    return organizations[:10]  # Limit to 10


def extract_abstract(header_text: str) -> str:
    """Extract the abstract section."""
    # Look for "Abstract" heading
    abstract_patterns = [
        r'Abstract\s*[:\-]?\s*(.*?)(?=\n\s*\n|\n\s*1\s*[\.\s]|Introduction|Keywords)',
        r'ABSTRACT\s*[:\-]?\s*(.*?)(?=\n\s*\n|\n\s*1\s*[\.\s]|INTRODUCTION|Keywords)',
    ]
    
    for pattern in abstract_patterns:
        match = re.search(pattern, header_text, re.DOTALL | re.IGNORECASE)
        if match:
            abstract = match.group(1).strip()
            # Clean up the abstract
            abstract = re.sub(r'\s+', ' ', abstract)
            if len(abstract) > 50:  # Minimum length check
                return abstract[:2000]  # Limit length
    
    return ""


def extract_section_headings(full_text: str) -> List[Dict[str, Any]]:
    """
    Extract section headings from the document.
    - Handles split headers (number on one line, title on next)
    - Filters out years, decimals, and reference entries
    - Keeps table captions as headings
    """
    
    acronym_exclusions = {
        'BLEU', 'GPU', 'CPU', 'LSTM', 'RNN', 'CNN', 'NLP', 'WMT', 'TPU',
        'ADAM', 'SGD', 'RELU', 'GELU', 'FFN', 'MLP', 'BERT', 'GPT', 'LLM',
        'PPL', 'GNMT', 'API', 'URL', 'HTTP', 'HTML', 'JSON', 'XML', 'SQL'
    }
    
    keyword_headings = {
        "abstract", "acknowledgements", "acknowledgments", "references",
        "appendix", "appendices"
    }
    
    def is_number_only(s: str) -> bool:
        return re.match(r"^\d+(?:\.\d+)*\.?$", s) is not None
    
    def is_valid_section_number(num: str) -> bool:
        if not re.match(r"^\d+(?:\.\d+)*$", num):
            return False
        parts = num.split(".")
        if len(parts) > 4:
            return False
        if len(parts[0]) > 2:  # reject 2014, 106, etc.
            return False
        if any(len(p) > 2 for p in parts[1:]):  # reject 3.200, 3.2.100
            return False
        return True
    
    def looks_like_title(title: str) -> bool:
        if len(title) < 3 or len(title) > 90:
            return False
        if not title[0].isalpha():
            return False
        if title.endswith(".") and any(c.islower() for c in title):
            return False
        alpha = sum(c.isalpha() for c in title)
        if alpha < 0.65 * len(title):
            return False
        return True
    
    def is_reference_entry(s: str) -> bool:
        return re.match(r"^\[\d+\]\s+", s) is not None
    
    def is_table_caption(s: str) -> bool:
        return re.match(r"^(Table|TABLE)\s+\d+\s*:\s+.+$", s) is not None
    
    # Normalize lines
    lines_raw = full_text.splitlines()
    lines = []
    for ln in lines_raw:
        s = re.sub(r"\s+", " ", (ln or "")).strip()
        if s:
            lines.append(s)
    
    sections = []
    in_references = False
    
    i = 0
    while i < len(lines):
        s = lines[i]
        
        # Skip reference entries once we hit references section
        if in_references:
            if is_reference_entry(s):
                i += 1
                continue
            i += 1
            continue
        
        # 1) Table captions
        if is_table_caption(s):
            m = re.match(r"^(Table|TABLE)\s+(?P<num>\d+)\s*:\s*(?P<title>.+)$", s)
            sections.append({
                "heading": s,
                "line_number": i,
                "number": f"Table {m.group('num')}",
                "title": f"Table {m.group('num')}: {m.group('title').strip()}",
                "pattern": "table_caption"
            })
            i += 1
            continue
        
        # 2) Merge split headers: "3.2.1" + next line title
        if is_number_only(s) and i + 1 < len(lines):
            nxt = lines[i + 1]
            if looks_like_title(nxt):
                s = f"{s.rstrip('.')} {nxt}"
                i += 1
        
        # 3) Skip reference entries
        if is_reference_entry(s):
            i += 1
            continue
        
        # 4) Keyword headings (Abstract, References, etc.)
        if s.strip().lower() in keyword_headings:
            title = s.strip()
            sections.append({
                "heading": title,
                "line_number": i,
                "number": None,
                "title": title,
                "pattern": "keyword_heading"
            })
            if title.lower() == "references":
                in_references = True
            i += 1
            continue
        
        # 5) ALL CAPS headings
        m_caps = re.match(r"^([A-Z]{3,}(?:\s+[A-Z]{3,})*)$", s)
        if m_caps:
            heading = m_caps.group(1)
            if heading not in acronym_exclusions and len(heading) >= 5:
                sections.append({
                    "heading": heading,
                    "line_number": i,
                    "number": None,
                    "title": heading,
                    "pattern": "all_caps"
                })
                if heading.lower() == "references":
                    in_references = True
                i += 1
                continue
        
        # 6) Numbered headings: "1 Introduction", "6.1 Machine Translation"
        m_num = re.match(r"^(?P<num>\d+(?:\.\d+)*)\.?\s+(?P<title>.+)$", s)
        if m_num:
            num = m_num.group("num")
            title = m_num.group("title").strip()
            
            if is_valid_section_number(num) and looks_like_title(title):
                sections.append({
                    "heading": f"{num} {title}",
                    "line_number": i,
                    "number": num,
                    "title": title,
                    "pattern": "numbered"
                })
                i += 1
                continue
        
        i += 1
    
    return sections


# -----------------------------------------------------------------------------
# TEXT EXTRACTION WITH STRUCTURE ANALYSIS
# -----------------------------------------------------------------------------

def extract_structured_text_from_pdf(pdf_path: str, structure: Dict[str, Any]) -> List[Document]:
    """
    Extract text with structure-aware metadata.
    Creates separate documents for:
    - Paper metadata (title, authors, etc.)
    - Abstract
    - Each major section
    """
    print("📄 Extracting structured text...")
    documents = []
    
    # Create metadata document
    if structure["title"] or structure["authors"]:
        metadata_text = f"## Paper Metadata\n\n"
        if structure["title"]:
            metadata_text += f"**Title:** {structure['title']}\n\n"
        if structure["authors"]:
            metadata_text += f"**Authors:** {', '.join(structure['authors'])}\n\n"
        if structure["emails"]:
            metadata_text += f"**Contact:** {', '.join(structure['emails'])}\n\n"
        if structure["organizations"]:
            metadata_text += f"**Affiliations:** {'; '.join(structure['organizations'])}\n\n"
        
        documents.append(Document(
            text=metadata_text,
            metadata={
                "source": pdf_path,
                "type": "metadata",
                "content_type": "metadata",
                "title": structure["title"],
                "authors": structure["authors"][:5] if structure["authors"] else [],
            }
        ))
    
    # Create abstract document
    if structure["abstract"]:
        abstract_text = f"## Abstract\n\n{structure['abstract']}"
        documents.append(Document(
            text=abstract_text,
            metadata={
                "source": pdf_path,
                "type": "abstract",
                "content_type": "section",
                "section_name": "Abstract",
            }
        ))
    
    # Extract text by sections
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text += page.get_text("text") + "\n\n"
    doc.close()
    
    # Split by detected sections
    if structure["sections"]:
        sections_with_content = extract_sections_content(full_text, structure["sections"])
        
        for section in sections_with_content:
            if section["content"] and len(section["content"]) > 50:
                section_text = f"## {section['heading']}\n\n{section['content']}"
                documents.append(Document(
                    text=clean_text(section_text),
                    metadata={
                        "source": pdf_path,
                        "type": "section",
                        "content_type": "section",
                        "section_name": section.get("title", section["heading"]),
                        "section_number": section.get("number", ""),
                    }
                ))
    
    print(f"   ✅ {len(documents)} structured documents created")
    return documents


def extract_sections_content(full_text: str, sections: List[Dict]) -> List[Dict]:
    """Extract content for each section."""
    lines = full_text.split('\n')
    sections_with_content = []
    
    for i, section in enumerate(sections):
        start_line = section["line_number"]
        
        # Find end (next section or end of document)
        if i + 1 < len(sections):
            end_line = sections[i + 1]["line_number"]
        else:
            end_line = len(lines)
        
        # Extract content
        content_lines = lines[start_line + 1:end_line]
        content = '\n'.join(content_lines)
        
        sections_with_content.append({
            **section,
            "content": content.strip()
        })
    
    return sections_with_content


# -----------------------------------------------------------------------------
# FULL TEXT EXTRACTION (ORIGINAL APPROACH - PRESERVED)
# -----------------------------------------------------------------------------

def extract_text_from_pdf(pdf_path: str) -> List[Document]:
    """Extract text from PDF using PyMuPDF (original approach)."""
    print("📄 Extracting full text (page by page)...")
    documents = []
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        
        if text.strip():
            cleaned = clean_text(text)
            if cleaned:
                documents.append(Document(
                    text=cleaned,
                    metadata={
                        "source": pdf_path,
                        "page": page_num + 1,
                        "type": "text",
                        "content_type": "text",
                    }
                ))
    
    doc.close()
    print(f"   ✅ {len(documents)} pages extracted")
    return documents


# -----------------------------------------------------------------------------
# REFERENCE EXTRACTION (NEW)
# -----------------------------------------------------------------------------

def extract_references_from_pdf(pdf_path: str) -> List[Document]:
    """
    Extract references/bibliography section from PDF.
    Creates separate documents for references.
    """
    print("📚 Extracting references...")
    
    doc = fitz.open(pdf_path)
    full_text = ""
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text += page.get_text("text") + "\n"
    
    doc.close()
    
    # Find references section
    ref_patterns = [
        r'(?:^|\n)\s*References?\s*\n',
        r'(?:^|\n)\s*REFERENCES?\s*\n',
        r'(?:^|\n)\s*Bibliography\s*\n',
        r'(?:^|\n)\s*BIBLIOGRAPHY\s*\n',
    ]
    
    ref_start = -1
    for pattern in ref_patterns:
        match = re.search(pattern, full_text, re.IGNORECASE)
        if match:
            ref_start = match.end()
            break
    
    if ref_start == -1:
        print("   ⚠️ No references section found")
        return []
    
    # Extract reference text (from "References" to end or next major section)
    ref_text = full_text[ref_start:]
    
    # Try to find where references end (Appendix, Supplementary, etc.)
    end_patterns = [
        r'\n\s*(?:Appendix|APPENDIX|Supplementary|SUPPLEMENTARY)',
    ]
    
    for pattern in end_patterns:
        match = re.search(pattern, ref_text)
        if match:
            ref_text = ref_text[:match.start()]
            break
    
    # Parse individual references
    references = parse_references(ref_text)
    
    if not references:
        # If parsing fails, just use the raw text
        if ref_text.strip():
            return [Document(
                text=f"## References\n\n{clean_text(ref_text)}",
                metadata={
                    "source": pdf_path,
                    "type": "references",
                    "content_type": "references",
                    "reference_count": 0,
                }
            )]
        return []
    
    # Create reference documents
    documents = []
    
    # Group references into chunks for better retrieval
    ref_chunks = []
    current_chunk = []
    current_length = 0
    
    for ref in references:
        ref_text = f"[{ref['number']}] {ref['text']}"
        ref_length = len(ref_text)
        
        if current_length + ref_length > config.REF_CHUNK_SIZE and current_chunk:
            ref_chunks.append(current_chunk)
            current_chunk = [ref_text]
            current_length = ref_length
        else:
            current_chunk.append(ref_text)
            current_length += ref_length
    
    if current_chunk:
        ref_chunks.append(current_chunk)
    
    for i, chunk in enumerate(ref_chunks):
        chunk_text = "## References\n\n" + "\n\n".join(chunk)
        documents.append(Document(
            text=chunk_text,
            metadata={
                "source": pdf_path,
                "type": "references",
                "content_type": "references",
                "chunk_index": i + 1,
                "total_chunks": len(ref_chunks),
                "reference_count": len(chunk),
            }
        ))
    
    print(f"   ✅ {len(references)} references extracted into {len(documents)} documents")
    return documents


def parse_references(ref_text: str) -> List[Dict[str, str]]:
    """Parse individual references from text."""
    references = []
    
    # Common reference patterns
    # Pattern 1: [1] Author, Title...
    # Pattern 2: 1. Author, Title...
    # Pattern 3: Author (Year). Title...
    
    # Try numbered references first
    numbered_pattern = r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|\Z)'
    matches = re.findall(numbered_pattern, ref_text, re.DOTALL)
    
    if matches:
        for num, text in matches:
            text = re.sub(r'\s+', ' ', text).strip()
            if len(text) > 20:  # Minimum length
                references.append({
                    "number": num,
                    "text": text
                })
        return references
    
    # Try period-numbered references
    period_pattern = r'(\d+)\.\s*(.*?)(?=\d+\.\s|\Z)'
    matches = re.findall(period_pattern, ref_text, re.DOTALL)
    
    if matches:
        for num, text in matches:
            text = re.sub(r'\s+', ' ', text).strip()
            if len(text) > 20:
                references.append({
                    "number": num,
                    "text": text
                })
        return references
    
    # Fallback: Split by common patterns
    lines = ref_text.split('\n')
    current_ref = ""
    ref_num = 1
    
    for line in lines:
        line = line.strip()
        if not line:
            if current_ref:
                references.append({
                    "number": str(ref_num),
                    "text": current_ref
                })
                ref_num += 1
                current_ref = ""
        else:
            current_ref += " " + line if current_ref else line
    
    if current_ref:
        references.append({
            "number": str(ref_num),
            "text": current_ref
        })
    
    return references


# -----------------------------------------------------------------------------
# TABLE EXTRACTION (UNCHANGED - KEEPING ORIGINAL STRATEGY)
# -----------------------------------------------------------------------------

def is_reference_table(text: str) -> bool:
    """
    Check if the extracted table is actually a references/bibliography section.
    Returns True if it looks like references (should be filtered out).
    """
    if not text.strip():
        return False
    
    reference_patterns = [
        r'arXiv preprint',
        r'arXiv:\d+\.\d+',
        r'In Proceedings',
        r'Journal of',
        r'et al\.',
        r'pages \d+[–-]\d+',
        r'In Advances in Neural',
        r'preprint arXiv',
    ]
    
    lines = text.split('\n')
    reference_matches = 0
    total_lines = 0
    
    for line in lines:
        line = line.strip()
        if not line or line == 'nan':
            continue
        total_lines += 1
        
        for pattern in reference_patterns:
            if re.search(pattern, line, re.IGNORECASE):
                reference_matches += 1
                break
    
    if total_lines > 0 and reference_matches / total_lines > 0.3:
        return True
    
    return False


def extract_tables_from_pdf(pdf_path: str) -> List[Document]:
    """
    Extract tables using hybrid approach (UNCHANGED):
    1. Tabula (Lattice mode) - for bordered tables
    2. Tabula (Stream mode) - for borderless tables
    3. PDFPlumber - alternative extraction
    4. PyMuPDF - built-in table detection
    """
    print("📊 Extracting tables (hybrid approach)...")
    
    all_table_texts = []
    references_filtered = 0
    
    # METHOD 1: TABULA LATTICE
    print("   🔷 Tabula Lattice...")
    try:
        tables_lattice = tabula.read_pdf(
            pdf_path,
            pages="all",
            multiple_tables=True,
            lattice=True,
            silent=True
        )
        
        for i, df in enumerate(tables_lattice):
            if not df.empty:
                table_text = df.to_markdown(index=False)
                
                if is_reference_table(table_text):
                    references_filtered += 1
                    continue
                
                if table_text.strip():
                    all_table_texts.append(f"[Tabula-Lattice Table {i+1}]\n{table_text}")
                    
        print(f"      Found {len(tables_lattice)} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # METHOD 2: TABULA STREAM
    print("   🔶 Tabula Stream...")
    try:
        tables_stream = tabula.read_pdf(
            pdf_path,
            pages="all",
            multiple_tables=True,
            stream=True,
            silent=True
        )
        
        for i, df in enumerate(tables_stream):
            if not df.empty:
                table_text = df.to_markdown(index=False)
                
                if is_reference_table(table_text):
                    references_filtered += 1
                    continue
                
                if table_text.strip():
                    all_table_texts.append(f"[Tabula-Stream Table {i+1}]\n{table_text}")
                    
        print(f"      Found {len(tables_stream)} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # METHOD 3: PDFPLUMBER
    print("   🔷 PDFPlumber...")
    try:
        with pdfplumber.open(pdf_path) as pdf:
            plumber_count = 0
            for page_num, page in enumerate(pdf.pages):
                tables = page.extract_tables()
                for i, table in enumerate(tables):
                    if table:
                        df = pd.DataFrame(table[1:], columns=table[0] if table[0] else None)
                        table_text = df.to_markdown(index=False)
                        
                        if is_reference_table(table_text):
                            references_filtered += 1
                            continue
                        
                        if table_text.strip():
                            all_table_texts.append(f"[PDFPlumber Page {page_num+1} Table {i+1}]\n{table_text}")
                            plumber_count += 1
            print(f"      Found {plumber_count} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # METHOD 4: PYMUPDF
    print("   🔶 PyMuPDF...")
    try:
        doc = fitz.open(pdf_path)
        pymupdf_count = 0
        for page_num in range(len(doc)):
            page = doc[page_num]
            tables = page.find_tables()
            for i, table in enumerate(tables):
                df = table.to_pandas()
                if not df.empty:
                    table_text = df.to_markdown(index=False)
                    
                    if is_reference_table(table_text):
                        references_filtered += 1
                        continue
                    
                    if table_text.strip():
                        all_table_texts.append(f"[PyMuPDF Page {page_num+1} Table {i+1}]\n{table_text}")
                        pymupdf_count += 1
        doc.close()
        print(f"      Found {pymupdf_count} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # Create documents
    table_docs = []
    for i, text in enumerate(all_table_texts):
        table_docs.append(Document(
            text=text,
            metadata={
                "type": "table",
                "content_type": "table",
                "table_index": i + 1
            }
        ))
    
    print(f"   📋 References filtered: {references_filtered}")
    print(f"   ✅ Total: {len(table_docs)} table documents")
    
    return table_docs




# -----------------------------------------------------------------------------
# IMAGE EXTRACTION (UNCHANGED)
# -----------------------------------------------------------------------------

def extract_images_from_pdf(pdf_path: str) -> List[Dict]:
    """Extract images from PDF."""
    print("🖼️ Extracting images...")
    images = []
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images(full=True)
        
        for img_idx, img in enumerate(image_list):
            try:
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                
                pil_image = Image.open(io.BytesIO(image_bytes))
                if pil_image.width < 100 or pil_image.height < 100:
                    continue
                
                images.append({
                    "page": page_num + 1,
                    "index": img_idx + 1,
                    "width": pil_image.width,
                    "height": pil_image.height,
                    "image": pil_image,
                    "bytes": image_bytes,
                })
            except Exception as e:
                pass
    
    doc.close()
    print(f"   ✅ {len(images)} images extracted")
    return images


def ocr_images(images: List[Dict]) -> List[Document]:
    """Run OCR on images using EasyOCR."""
    if not images:
        return []
    
    print("🔍 Running OCR...")
    import easyocr
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)
    
    ocr_docs = []
    for img_data in images:
        try:
            image = img_data["image"]
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            results = reader.readtext(np.array(image))
            text = " ".join([r[1] for r in results])
            
            if text.strip():
                ocr_docs.append(Document(
                    text=f"Figure (page {img_data['page']}): {clean_text(text)}",
                    metadata={
                        "type": "image_ocr",
                        "content_type": "figure",
                        "page": img_data["page"]
                    }
                ))
        except:
            pass
    
    print(f"   ✅ OCR completed for {len(ocr_docs)} images")
    return ocr_docs


def create_image_descriptions(images: List[Dict]) -> List[Document]:
    """Create text descriptions for images using Gemini Vision."""
    if not images:
        return []
    
    print("🎨 Creating image descriptions with Vision AI...")
    
    try:
        vision_client = genai.Client(api_key=config.GOOGLE_API_KEY)
        
        docs = []
        for idx, img_data in enumerate(images, 1):
            try:
                print(f"   Processing {idx}/{len(images)} (page {img_data['page']})...", end=" ")
                
                img_buffer = io.BytesIO()
                img_data["image"].save(img_buffer, format='PNG')
                img_bytes = img_buffer.getvalue()
                
                prompt = """Analyze this scientific figure/diagram and provide a detailed description.

Focus on:
1. Figure number and Type of diagram (architecture, flowchart, graph, table, etc.)
2. IMPORTANT: Identify the number of independent diagrams and their positions in the image if multiple are present.
3. Main components and their relationships.
4. Any text labels, equations, or annotations.
5. Key insights or patterns shown.

Be specific and technical. This description will be used for RAG retrieval."""
                
                response = vision_client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=[
                        types.Content(
                            role="user",
                            parts=[
                                types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
                                types.Part.from_text(text=prompt),
                            ],
                        ),
                    ],
                )
                
                description = response.text
                
                if description:
                    docs.append(Document(
                        text=f"Figure from page {img_data['page']}: {description}",
                        metadata={
                            "type": "image_description",
                            "content_type": "figure",
                            "page": img_data["page"]
                        }
                    ))
                    print("✅")
                else:
                    print("⚠️ No description")
                    
            except Exception as e:
                print(f"❌ Error: {str(e)[:50]}")
        
        print(f"   ✅ {len(docs)} descriptions created")
        return docs
        
    except Exception as e:
        print(f"   ❌ Vision API error: {e}")
        return []

In [50]:
# =============================================================================
# DEBUG CELL: Inspect Section Extraction
# =============================================================================
# Run this cell to see what sections are being detected

import fitz
import re

import re
import fitz

import re
import fitz

def debug_section_extraction(pdf_path: str):
    """
    Debug function to inspect section extraction results.
    - Robust to PDF headers split across lines (e.g., "3.2.1" on one line and title next).
    - Keeps table captions (Table 1:, Table 2:, ...) as headings.
    - Reduces noise from years, decimals, and reference entries.
    """
    doc = fitz.open(pdf_path)

    # Get full text with real newlines
    pages_text = []
    for page_num in range(len(doc)):
        pages_text.append(doc[page_num].get_text("text"))
    doc.close()

    text = "\n".join(pages_text)
    lines_raw = text.splitlines()

    # Normalize whitespace and drop empties
    lines = []
    for ln in lines_raw:
        s = re.sub(r"\s+", " ", (ln or "")).strip()
        if s:
            lines.append(s)

    acronym_exclusions = {
        'BLEU', 'GPU', 'CPU', 'LSTM', 'RNN', 'CNN', 'NLP', 'WMT', 'TPU',
        'ADAM', 'SGD', 'RELU', 'GELU', 'FFN', 'MLP', 'BERT', 'GPT', 'LLM',
        'PPL', 'GNMT', 'API', 'URL', 'HTTP', 'HTML', 'JSON', 'XML', 'SQL'
    }

    def is_number_only(s: str) -> bool:
        return re.match(r"^\d+(?:\.\d+)*\.?$", s) is not None

    def is_valid_section_number(num: str) -> bool:
        # Accept: 1, 2, 6.1, 3.2.1, ...
        # Reject: 2014, 106, 23.75 (decimals / years / table values)
        if not re.match(r"^\d+(?:\.\d+)*$", num):
            return False
        parts = num.split(".")
        if len(parts) > 4:
            return False
        # reject 3+ digit top-level like 2014
        if len(parts[0]) > 2:
            return False
        # reject deep parts like 3.200 or 3.2.100
        if any(len(p) > 2 for p in parts[1:]):
            return False
        return True

    def looks_like_title(title: str) -> bool:
        if len(title) < 3 or len(title) > 90:
            return False
        # Must start with a letter (filters "√dk", "[21]", "2014 English-...", etc.)
        if not title[0].isalpha():
            return False
        # Avoid prose-y lines
        if title.endswith(".") and any(c.islower() for c in title):
            return False
        alpha = sum(c.isalpha() for c in title)
        if alpha < 0.65 * len(title):
            return False
        return True

    def is_reference_entry(s: str) -> bool:
        return re.match(r"^\[\d+\]\s+", s) is not None

    def is_table_caption(s: str) -> bool:
        return re.match(r"^(Table|TABLE)\s+\d+\s*:\s+.+$", s) is not None

    # Common unnumbered headings (mixed case)
    keyword_headings = {
        "abstract", "acknowledgements", "acknowledgments", "references",
        "appendix", "appendices"
    }

    sections = []
    in_references = False

    i = 0
    while i < len(lines):
        s = lines[i]

        # Once we hit references, optionally stop to avoid pulling [21]... etc.
        if in_references:
            # Keep the "References" heading only; skip individual refs
            if is_reference_entry(s):
                i += 1
                continue
            # You can break here if you don't want anything after references
            i += 1
            continue

        # 1) Keep Table captions explicitly (as headings)
        if is_table_caption(s):
            m = re.match(r"^(Table|TABLE)\s+(?P<num>\d+)\s*:\s*(?P<title>.+)$", s)
            sections.append({
                "line_number": i,
                "heading": s,
                "section_number": f"Table {m.group('num')}",
                "section_title": f"Table {m.group('num')}: {m.group('title').strip()}",
                "pattern": "table_caption"
            })
            i += 1
            continue

        # 2) Merge split headers: "3.2.1" (line i) + "Scaled Dot-Product Attention" (line i+1)
        if is_number_only(s) and i + 1 < len(lines):
            nxt = lines[i + 1]
            # also allow table captions split like "Table 1:" + "...", though less common
            if looks_like_title(nxt):
                s_merged = f"{s.rstrip('.')} {nxt}"
                s = s_merged
                i += 1  # consume next line

        # 3) Reject obvious ref entries early
        if is_reference_entry(s):
            i += 1
            continue

        # 4) Unnumbered keyword headings (Abstract, References, etc.)
        if s.strip().lower() in keyword_headings:
            title = s.strip()
            sections.append({
                "line_number": i,
                "heading": title,
                "section_number": None,
                "section_title": title,
                "pattern": "keyword_heading"
            })
            if title.lower() == "references":
                in_references = True
            i += 1
            continue

        # 5) ALL CAPS headings (e.g., ABSTRACT / REFERENCES) with acronym exclusion
        m_caps = re.match(r"^([A-Z]{3,}(?:\s+[A-Z]{3,})*)$", s)
        if m_caps:
            heading = m_caps.group(1)
            if heading not in acronym_exclusions and len(heading) >= 5:
                sections.append({
                    "line_number": i,
                    "heading": heading,
                    "section_number": None,
                    "section_title": heading,
                    "pattern": "all_caps"
                })
                if heading.lower() == "references":
                    in_references = True
                i += 1
                continue

        # 6) Numbered headings: 1 Introduction / 6.1 Machine Translation / 3.2.1 ...
        m_num = re.match(r"^(?P<num>\d+(?:\.\d+)*)\.?\s+(?P<title>.+)$", s)
        if m_num:
            num = m_num.group("num")
            title = m_num.group("title").strip()

            # extra guard against decimals masquerading as "num" (rare but safe)
            if "." in num:
                # ensure all dot-separated components are digits and short
                if not is_valid_section_number(num):
                    i += 1
                    continue
            else:
                if not is_valid_section_number(num):
                    i += 1
                    continue

            if looks_like_title(title):
                sections.append({
                    "line_number": i,
                    "heading": f"{num} {title}",
                    "section_number": num,
                    "section_title": title,
                    "pattern": "numbered"
                })
                i += 1
                continue

        i += 1

    # Pretty print
    print("=" * 80)
    print("SECTION DETECTION RESULTS")
    print("=" * 80)
    print(f"Total lines in document: {len(lines)}\n")

    numbered = [s for s in sections if s["pattern"] == "numbered"]
    tables = [s for s in sections if s["pattern"] == "table_caption"]
    caps = [s for s in sections if s["pattern"] == "all_caps"]
    keywords = [s for s in sections if s["pattern"] == "keyword_heading"]

    print(f"Numbered sections found: {len(numbered)}")
    print("-" * 40)
    for s in numbered:
        print(f"  Line {s['line_number']:4d}: [{s['section_number']}] {s['section_title']}")

    print(f"\nTable captions found: {len(tables)}")
    print("-" * 40)
    for s in tables:
        print(f"  Line {s['line_number']:4d}: [{s['section_number']}] {s['section_title']}")

    print(f"\nALL CAPS headings found: {len(caps)}")
    print("-" * 40)
    for s in caps:
        print(f"  Line {s['line_number']:4d}: {s['section_title']}")

    print(f"\nKeyword headings found: {len(keywords)}")
    print("-" * 40)
    for s in keywords:
        print(f"  Line {s['line_number']:4d}: {s['section_title']}")

    print("\n" + "=" * 80)
    print("ALL SECTIONS (in order)")
    print("=" * 80)
    for idx, s in enumerate(sections, 1):
        num = s["section_number"] if s["section_number"] else "---"
        print(f"{idx:3d}. [Line {s['line_number']:4d}] [{num:>6}] {s['section_title']}")

    return sections




# Run the debug
sections = debug_section_extraction(config.PDF_PATH)

SECTION DETECTION RESULTS
Total lines in document: 3116

Numbered sections found: 137
----------------------------------------
  Line   33: [1] Introduction
  Line  100: [2] Related Work
  Line  102: [2.1] Memory for LLM Agents
  Line  113: [2] LLM Agents
  Line  208: [2.2] Retrieval-Augmented Generation
  Line  227: [3] Methodolodgy
  Line  234: [3.1] Note Construction
  Line  262: [3.2] Link Generation
  Line  290: [3.3] Memory Evolution
  Line  294: [4] neighbor set Mn
  Line  309: [3.4] Retrieve Relative Memory
  Line  329: [4] Experiment
  Line  331: [4.1] Dataset and Evaluation
  Line  817: [4.2] Implementation Details
  Line  827: [4.3] Empricial Results
  Line  857: [15.76] MemGPT
  Line  864: [8.54] A-MEM
  Line  902: [18.02] w/o ME
  Line  913: [45.33] A-MEM
  Line  945: [4.4] Ablation Study
  Line  959: [4.5] Hyperparameter Analysis
  Line  972: [50] k values
  Line  998: [50] k values
  Line 1023: [50] k values
  Line 1047: [50] k values
  Line 1071: [50] k values
  Line 11

## 8. Run Document Collection

In [51]:
# =============================================================================
# SECTION 8: DOCUMENT COLLECTION PIPELINE (UPDATED)
# =============================================================================

def collect_all_documents(pdf_path: str) -> List[Document]:
    """
    Complete document collection pipeline with structure analysis.
    
    Flow:
    1. Structure Analysis (title, authors, emails, organizations, abstract, sections)
    2. Structured Text Extraction (metadata, abstract, sections as separate docs)
    3. Full Text Extraction (original page-by-page approach)
    4. Table Extraction (unchanged hybrid approach)
    5. Image Extraction + OCR (unchanged)
    6. Image Descriptions (unchanged)
    7. Reference Extraction (NEW)
    """
    print("\n" + "="*60)
    print("DOCUMENT COLLECTION PIPELINE (ENHANCED)")
    print("="*60)
    
    all_docs = []
    
    # Step 1: Structure Analysis
    structure = analyze_document_structure(pdf_path)
    
    # Step 2: Structured Text Extraction (with metadata)
    structured_docs = extract_structured_text_from_pdf(pdf_path, structure)
    all_docs.extend(structured_docs)
    
    # # # Step 3: Full Text Extraction (original approach)
    text_docs = extract_text_from_pdf(pdf_path)
    all_docs.extend(text_docs)
    
    # Step 4: Table Extraction (unchanged)
    table_docs = extract_tables_from_pdf(pdf_path)
    all_docs.extend(table_docs)
    
    # Step 5: Image Extraction
    images = extract_images_from_pdf(pdf_path)
    
    # Step 6: OCR (unchanged)
    ocr_docs = ocr_images(images)
    all_docs.extend(ocr_docs)
    
    # Step 7: Image Descriptions (unchanged)
    img_desc_docs = create_image_descriptions(images)
    all_docs.extend(img_desc_docs)
    
    # Step 8: Reference Extraction (NEW)
    ref_docs = extract_references_from_pdf(pdf_path)
    all_docs.extend(ref_docs)
    
    # Summary
    print("\n" + "-"*40)
    print("📊 DOCUMENT COLLECTION SUMMARY:")
    print(f"   📝 Structured docs: {len(structured_docs)}")
    # print(f"   📄 Text pages: {len(text_docs)}")
    print(f"   📊 Tables: {len(table_docs)}")
    print(f"   🔍 OCR docs: {len(ocr_docs)}")
    print(f"   🎨 Image descriptions: {len(img_desc_docs)}")
    print(f"   📚 Reference docs: {len(ref_docs)}")
    print(f"\n📚 Total: {len(all_docs)} documents")
    print(img_desc_docs)
    return all_docs
# Run collection
all_documents = collect_all_documents(config.PDF_PATH)


DOCUMENT COLLECTION PIPELINE (ENHANCED)
📋 Analyzing document structure...
   📌 Title: arXiv:2502.12110v11 [cs.CL] 8 Oct 2025
   👥 Authors: 0 found
   📧 Emails: 1 found
   🏢 Organizations: 2 found
   📝 Abstract: Found
   📑 Sections: 179 found
📄 Extracting structured text...
   ✅ 165 structured documents created
📄 Extracting full text (page by page)...
   ✅ 28 pages extracted
📊 Extracting tables (hybrid approach)...
   🔷 Tabula Lattice...
      Found 84 tables
   🔶 Tabula Stream...
      Found 17 tables
   🔷 PDFPlumber...
      Found 18 tables
   🔶 PyMuPDF...
      Found 9 tables
   📋 References filtered: 2
   ✅ Total: 122 table documents
🖼️ Extracting images...
   ✅ 2 images extracted
🔍 Running OCR...


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


   ✅ OCR completed for 0 images
🎨 Creating image descriptions with Vision AI...
   Processing 1/2 (page 3)... ✅
   Processing 2/2 (page 3)... ✅
   ✅ 2 descriptions created
📚 Extracting references...
   ✅ 39 references extracted into 14 documents

----------------------------------------
📊 DOCUMENT COLLECTION SUMMARY:
   📝 Structured docs: 165
   📊 Tables: 122
   🔍 OCR docs: 0
   🎨 Image descriptions: 2
   📚 Reference docs: 14

📚 Total: 331 documents
[Document(id_='f30555c4-75d1-4ab0-8926-4ea3ba75cccd', embedding=None, metadata={'type': 'image_description', 'content_type': 'figure', 'page': 3}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Figure from page 3: Here is a detailed, technical description of the provided scientific figure/diagram:\n\n1.  **Figure Number and Type of Diagram:**\n    *   No figure number is present in the

## 9. Create Ingestion Pipeline

In [10]:
print("\n" + "="*60)
print("INGESTION PIPELINE")
print("="*60)

def create_optimized_chunking(documents: List[Document]) -> List:
    """
    Create optimized chunking strategy based on content type.
    Uses SentenceSplitter with paragraph_separator="\\n\\n" for all types.
    
    Chunk sizes:
    - Text/Sections: 400 tokens, 80 overlap
    - Tables: 1400 tokens, 50 overlap
    - Figures: 500 tokens, 80 overlap
    - References: 700 tokens, 50 overlap
    
    Args:
        documents: List of Document objects with content_type metadata
    
    Returns:
        List of nodes ready for embedding
    """
    print("\n" + "="*60)
    print("OPTIMIZED CHUNKING PIPELINE")
    print("="*60)
    
    from llama_index.core.schema import TextNode
    
    all_nodes = []
    
    # Track counts by type
    type_counts = {
        "text": 0,
        "section": 0,
        "metadata": 0,
        "table": 0,
        "figure": 0,
        "references": 0,
     }
    
    for doc in documents:
        content_type = doc.metadata.get("content_type", "text")
        doc_type = doc.metadata.get("type", "text")
        
        # Determine chunking strategy based on content type
        if content_type == "table" or doc_type == "table":
            # Large chunks for tables to preserve structure
            splitter = SentenceSplitter(
                chunk_size=config.TABLE_CHUNK_SIZE,
                chunk_overlap=config.TABLE_CHUNK_OVERLAP,
                paragraph_separator="\n\n"
            )
            nodes = splitter.get_nodes_from_documents([doc])
            type_counts["table"] += len(nodes)
            
        elif content_type == "figure" or doc_type in ["image_ocr", "image_description"]:
            # Medium chunks for figures
            splitter = SentenceSplitter(
                chunk_size=config.FIGURE_CHUNK_SIZE,
                chunk_overlap=config.FIGURE_CHUNK_OVERLAP,
                paragraph_separator="\n\n"
            )
            nodes = splitter.get_nodes_from_documents([doc])
            type_counts["figure"] += len(nodes)
            
        elif content_type == "references" or doc_type == "references":
            # Dedicated chunk size for references
            splitter = SentenceSplitter(
                chunk_size=config.REF_CHUNK_SIZE,
                chunk_overlap=config.REF_CHUNK_OVERLAP,
                paragraph_separator="\n\n"
            )
            nodes = splitter.get_nodes_from_documents([doc])
            type_counts["references"] += len(nodes)
            
        elif content_type == "metadata" or doc_type == "metadata":
            # Keep metadata as single node (usually small)
            nodes = [TextNode(
                text=doc.text,
                metadata={
                    **doc.metadata,
                    "chunk_id": 0,
                    "total_chunks": 1,
                }
            )]
            type_counts["metadata"] += len(nodes)
            
        elif content_type == "section" or doc_type == "section":

            splitter = SentenceSplitter(
            chunk_size=config.SEC_CHUNK_SIZE,
            chunk_overlap=config.SEC_CHUNK_OVERLAP,
            paragraph_separator="\n\n"
             )
            nodes = splitter.get_nodes_from_documents([doc])
        
            type_counts["section"] += len(nodes)
        else:
            splitter = SentenceSplitter(
            chunk_size=config.TEXT_CHUNK_SIZE,
            chunk_overlap=config.TEXT_CHUNK_OVERLAP,
            paragraph_separator="\n\n"
             )
            nodes = splitter.get_nodes_from_documents([doc])

            type_counts["text"] += len(nodes)
        
        # Ensure all nodes have content_type metadata
        for node in nodes:
            if "content_type" not in node.metadata:
                node.metadata["content_type"] = content_type
        
        all_nodes.extend(nodes)
    
    # Print summary
    print("\n📊 CHUNKING SUMMARY:")
    print(f"   📄 Text nodes: {type_counts['text']}")
    print(f"   📑 Section nodes: {type_counts['section']}")
    print(f"   📌 Metadata nodes: {type_counts['metadata']}")
    print(f"   📊 Table nodes: {type_counts['table']}")
    print(f"   🖼️ Figure nodes: {type_counts['figure']}")
    print(f"   📚 Reference nodes: {type_counts['references']}")
    print(f"\n✅ Total nodes created: {len(all_nodes)}")
    
    return all_nodes


# =============================================================================
# SECTION 9 (ALTERNATIVE): INGESTION PIPELINE WITH OPTIMIZED CHUNKING
# =============================================================================

def run_optimized_ingestion_pipeline(
    documents: List[Document],
    embed_model
) -> List:
    """
    Run the complete ingestion pipeline with optimized chunking.
    
    This replaces the original ingestion pipeline in the notebook.
    
    Args:
        documents: List of documents from collect_all_documents()
        embed_model: The embedding model instance
    
    Returns:
        List of embedded nodes ready for indexing
    """
    print("\n" + "="*60)
    print("INGESTION PIPELINE (OPTIMIZED)")
    print("="*60)
    
    # Step 1: Create optimized chunks
    nodes = create_optimized_chunking(documents)
    
    # Step 2: Apply embeddings
    print("\n🔄 Generating embeddings...")
    pipeline = IngestionPipeline(transformations=[embed_model])
    nodes = pipeline.run(nodes=nodes, show_progress=True)
    
    print(f"\n✅ Created {len(nodes)} embedded nodes")
    
    return nodes


nodes = run_optimized_ingestion_pipeline(
    all_documents,
    embed_model,
)

print(f"\n✅ Created {len(nodes)} nodes")


INGESTION PIPELINE

INGESTION PIPELINE (OPTIMIZED)

OPTIMIZED CHUNKING PIPELINE

📊 CHUNKING SUMMARY:
   📄 Text nodes: 23
   📑 Section nodes: 55
   📌 Metadata nodes: 1
   📊 Table nodes: 6
   🖼️ Figure nodes: 13
   📚 Reference nodes: 9

✅ Total nodes created: 107

🔄 Generating embeddings...


Generating embeddings: 100%|██████████| 107/107 [00:17<00:00,  6.13it/s]


✅ Created 107 embedded nodes

✅ Created 107 nodes


## 10. Create Vector Index

In [11]:
print("\n" + "="*60)
print("CREATING VECTOR INDEX")
print("="*60)



index = VectorStoreIndex(
    nodes=nodes,
    embed_model=embed_model,
    show_progress=True,
)

print(f"✅ Index created with {len(nodes)} nodes")


CREATING VECTOR INDEX


Generating embeddings: 0it [00:00, ?it/s]

✅ Index created with 107 nodes


## 11. Create Hybrid Fusion Retriever

In [27]:
print("\n" + "="*60)
print("CREATING RETRIEVERS")
print("="*60)


# Vector retriever
vector_retriever = index.as_retriever(similarity_top_k=50)

# BM25 retriever
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=50,
    stemmer=Stemmer.Stemmer("english"),
    language="english",
)

# Hybrid Fusion with Reciprocal Rank Fusion
hybrid_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    retriever_weights=[config.VECTOR_WEIGHT, config.BM25_WEIGHT],
    similarity_top_k=50,
    num_queries=1,
    mode="reciprocal_rerank",  
    use_async=True,
    verbose=False,
)

print(f"✅ Vector Retriever (weight: {config.VECTOR_WEIGHT})")
print(f"✅ BM25 Retriever (weight: {config.BM25_WEIGHT})")
print(f"✅ Hybrid Fusion with Reciprocal Rank Fusion")


CREATING RETRIEVERS
✅ Vector Retriever (weight: 0.8)
✅ BM25 Retriever (weight: 0.2)
✅ Hybrid Fusion with Reciprocal Rank Fusion


In [28]:
print("\n" + "="*60)
print("INITIALIZING RERANKER")
print("="*60)

# Initialize the cross-encoder reranker for second-stage ranking
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=10  # Final number of results after reranking
)

print("✅ Reranker: cross-encoder/ms-marco-MiniLM-L-2-v2 (top_n=5)")


INITIALIZING RERANKER
✅ Reranker: cross-encoder/ms-marco-MiniLM-L-2-v2 (top_n=5)


## 12. Create Query Engines

In [29]:
# Query engines WITH reranking
vector_engine = RetrieverQueryEngine.from_args(
    vector_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

bm25_engine = RetrieverQueryEngine.from_args(
    bm25_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

hybrid_engine = RetrieverQueryEngine.from_args(
    hybrid_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

print("✅ Query engines created with reranking")

✅ Query engines created with reranking


## 13. Evaluation Questions

In [15]:
# Load questions from file
with open("transformer_gold_qa_50.txt", "r") as f:
    eval_questions = [line.strip() for line in f if line.strip()]

print(f"✅ Loaded {len(eval_questions)} questions")
print(f"\n📝 First 5 questions:")
for i, q in enumerate(eval_questions[:5], 1):
    print(f"   {i}. {q}...")

print(f"✅ {len(eval_questions)} evaluation questions")

✅ Loaded 50 questions

📝 First 5 questions:
   1. According to Table 2, what was the estimated training cost in FLOPs for the 'Transformer (big)' model on the English-to-French task?...
   2. Looking at Table 3, what was the development set BLEU score for the base model after 100,000 steps?...
   3. Based on Table 1, which layer type has a 'Maximum Path Length' of O(log_k(n))?...
   4. From Table 2, compare the EN-DE BLEU scores of the 'GNMT RL Ensemble' and the 'Transformer (base model)'....
   5. According to Table 3, row A, how does changing the number of heads (h) to 1 affect the BLEU score compared to the base model?...
✅ 50 evaluation questions


In [16]:
print("\n" + "="*60)
print("GENERATING QA DATASET FOR RETRIEVER EVALUATION")
print("="*60)

from llama_index.core.evaluation import generate_question_context_pairs

# Generate question-context pairs from nodes
qa_dataset = generate_question_context_pairs(
    nodes=nodes,
    llm=llm,
    num_questions_per_chunk=1  # 2 questions per chunk
)

# Check the structure
print(f"✅ Generated QA dataset")
print(f"   Queries: {len(qa_dataset.queries)}")
print(f"   Relevant docs: {len(qa_dataset.relevant_docs)}")

# Show sample - queries is a dict, not a list
if qa_dataset.queries:
    # Get first query ID
    first_query_id = list(qa_dataset.queries.keys())[0]
    first_query_text = qa_dataset.queries[first_query_id]
    
    print(f"\n📝 Sample question:")
    print(f"   ID: {first_query_id}")
    print(f"   Text: {first_query_text}")
    
    # Show relevant docs for this query
    if first_query_id in qa_dataset.relevant_docs:
        relevant_node_ids = qa_dataset.relevant_docs[first_query_id]
        print(f"   Relevant docs: {len(relevant_node_ids)} nodes")
else:
    print("\n No questions generated!")
    print("   This might happen if:")
    print("   - Dataset is too small")
    print("   - LLM failed to generate questions")
    print("   - Try reducing num_questions_per_chunk to 1")



GENERATING QA DATASET FOR RETRIEVER EVALUATION


100%|██████████| 107/107 [09:16<00:00,  5.20s/it]

✅ Generated QA dataset
   Queries: 107
   Relevant docs: 107

📝 Sample question:
   ID: 9e44a9ca-87bf-4fe0-b2e8-d6f8ec3cb147
   Text: Based on the provided metadata for the paper "Attention Is All You Need", identify all authors who are affiliated with Google Research and have their contact email explicitly listed.
   Relevant docs: 1 nodes


## 14. Run Evaluation

In [30]:
from llama_index.core.evaluation import RetrieverEvaluator

# Initialize evaluators
print("\n" + "="*60)
print("INITIALIZING EVALUATORS")
print("="*60)

faithfulness_eval = FaithfulnessEvaluator(llm=judge_llm)
relevancy_eval = RelevancyEvaluator(llm=judge_llm)

# Retriever evaluators for MRR, Hit Rate, Precision, Recall
vector_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=vector_retriever
)

bm25_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=bm25_retriever
)

hybrid_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=hybrid_retriever
)

print("✅ Faithfulness & Relevancy evaluators initialized")
print("✅ Retriever evaluators initialized (MRR, Hit Rate, Precision, Recall)")


# ============================================================================
# PART 1: RETRIEVER METRICS (MRR, Hit Rate, Precision, Recall)
# ============================================================================

async def evaluate_retrievers():
    """Evaluate retrievers on qa_dataset."""
    print("\n" + "="*60)
    print("RETRIEVER METRICS ON QA DATASET")
    print("="*60)
    
    # Evaluate each retriever
    print("\n📊 Evaluating Vector Retriever...")
    vector_results = await vector_retriever_eval.aevaluate_dataset(qa_dataset)
    
    print("📊 Evaluating BM25 Retriever...")
    bm25_results = await bm25_retriever_eval.aevaluate_dataset(qa_dataset)
    
    print("📊 Evaluating Hybrid Retriever...")
    hybrid_results = await hybrid_retriever_eval.aevaluate_dataset(qa_dataset)
    
    # Extract metrics
    def get_avg_metrics(results):
        metric_dicts = [r.metric_vals_dict for r in results]
        df = pd.DataFrame(metric_dicts)
        return df.mean()
    
    vector_metrics = get_avg_metrics(vector_results)
    bm25_metrics = get_avg_metrics(bm25_results)
    hybrid_metrics = get_avg_metrics(hybrid_results)
    
    # Create comparison table
    retriever_comparison = pd.DataFrame({
        "Retriever": ["Vector", "BM25", "Hybrid (RRF)"],
        "MRR": [
            vector_metrics["mrr"],
            bm25_metrics["mrr"],
            hybrid_metrics["mrr"]
        ],
        "Hit Rate": [
            vector_metrics["hit_rate"],
            bm25_metrics["hit_rate"],
            hybrid_metrics["hit_rate"]
        ],
        "Precision": [
            vector_metrics["precision"],
            bm25_metrics["precision"],
            hybrid_metrics["precision"]
        ],
        "Recall": [
            vector_metrics["recall"],
            bm25_metrics["recall"],
            hybrid_metrics["recall"]
        ],
    })
    
    print("\n" + "="*60)
    print("RETRIEVER PERFORMANCE COMPARISON")
    print("="*60)
    print(retriever_comparison.to_string(index=False))
    
    return retriever_comparison


# ============================================================================
# PART 2: FAITHFULNESS & RELEVANCY ON EVAL QUESTIONS
# ============================================================================

async def evaluate_rag_quality():
    """Evaluate faithfulness and relevancy on eval_questions."""
    print("\n" + "="*60)
    print("RAG QUALITY EVALUATION ON EVAL QUESTIONS")
    print("="*60)
    
    results = []
    
    for i, q in enumerate(eval_questions, 1):
        print(f"\n🔍 Q{i}: {q[:50]}...")
        
        # Get responses (async)
        vec_resp = await vector_engine.aquery(q)
        bm25_resp = await bm25_engine.aquery(q)
        hybrid_resp = await hybrid_engine.aquery(q)
        
        # Evaluate (async)
        vec_faith = await faithfulness_eval.aevaluate_response(response=vec_resp)
        bm25_faith = await faithfulness_eval.aevaluate_response(response=bm25_resp)
        hybrid_faith = await faithfulness_eval.aevaluate_response(response=hybrid_resp)
        
        vec_rel = await relevancy_eval.aevaluate_response(query=q, response=vec_resp)
        bm25_rel = await relevancy_eval.aevaluate_response(query=q, response=bm25_resp)
        hybrid_rel = await relevancy_eval.aevaluate_response(query=q, response=hybrid_resp)
        
        results.append({
            "question": q,
            "vector_faith": vec_faith.score,
            "vector_rel": vec_rel.score,
            "bm25_faith": bm25_faith.score,
            "bm25_rel": bm25_rel.score,
            "hybrid_faith": hybrid_faith.score,
            "hybrid_rel": hybrid_rel.score,
            "hybrid_response": str(hybrid_resp.response)[:200],
        })
        
        print(f"   Vector: F={vec_faith.score:.2f} R={vec_rel.score:.2f}")
        print(f"   BM25:   F={bm25_faith.score:.2f} R={bm25_rel.score:.2f}")
        print(f"   Hybrid: F={hybrid_faith.score:.2f} R={hybrid_rel.score:.2f}")
    
    return pd.DataFrame(results)


# ============================================================================
# RUN BOTH EVALUATIONS
# ============================================================================

# Run retriever evaluation
retriever_df = await evaluate_retrievers()

# Run RAG quality evaluation
rag_quality_df = await evaluate_rag_quality()


INITIALIZING EVALUATORS
✅ Faithfulness & Relevancy evaluators initialized
✅ Retriever evaluators initialized (MRR, Hit Rate, Precision, Recall)

RETRIEVER METRICS ON QA DATASET

📊 Evaluating Vector Retriever...
📊 Evaluating BM25 Retriever...
📊 Evaluating Hybrid Retriever...

RETRIEVER PERFORMANCE COMPARISON
   Retriever      MRR  Hit Rate  Precision   Recall
      Vector 0.562879  0.962617   0.019252 0.962617
        BM25 0.495090  0.915888   0.018318 0.915888
Hybrid (RRF) 0.565633  0.953271   0.019065 0.953271

RAG QUALITY EVALUATION ON EVAL QUESTIONS

🔍 Q1: According to Table 2, what was the estimated train...
   Vector: F=0.00 R=0.00
   BM25:   F=0.00 R=1.00
   Hybrid: F=1.00 R=1.00

🔍 Q2: Looking at Table 3, what was the development set B...
   Vector: F=1.00 R=1.00
   BM25:   F=1.00 R=1.00
   Hybrid: F=1.00 R=1.00

🔍 Q3: Based on Table 1, which layer type has a 'Maximum ...
   Vector: F=1.00 R=1.00
   BM25:   F=1.00 R=1.00
   Hybrid: F=1.00 R=1.00

🔍 Q4: From Table 2, compare the

## 15. Results Summary

In [31]:
print("\n" + "="*60)
print("COMPLETE EVALUATION SUMMARY")
print("="*60)

# ==========================================
# RETRIEVER METRICS SUMMARY
# ==========================================
print("\n📊 RETRIEVER METRICS:")
print(retriever_df.to_string(index=False))

# Find best retriever for each metric
print("\n🏆 BEST PERFORMERS:")
for metric in ["MRR", "Hit Rate", "Precision", "Recall"]:
    best_idx = retriever_df[metric].idxmax()
    best_retriever = retriever_df.loc[best_idx, "Retriever"]
    best_score = retriever_df.loc[best_idx, metric]
    print(f"   {metric}: {best_retriever} ({best_score:.4f})")

# ==========================================
# RAG QUALITY SUMMARY
# ==========================================
print("\n📊 RAG QUALITY METRICS:")
quality_summary = {
    "Retriever": ["Vector", "BM25", "Hybrid (RRF)"],
    "Avg Faithfulness": [
        rag_quality_df["vector_faith"].mean(),
        rag_quality_df["bm25_faith"].mean(),
        rag_quality_df["hybrid_faith"].mean(),
    ],
    "Avg Relevancy": [
        rag_quality_df["vector_rel"].mean(),
        rag_quality_df["bm25_rel"].mean(),
        rag_quality_df["hybrid_rel"].mean(),
    ],
}

quality_df = pd.DataFrame(quality_summary)
print(quality_df.to_string(index=False))

# ==========================================
# SAVE RESULTS
# ==========================================
os.makedirs("./results", exist_ok=True)
retriever_df.to_csv("./results/retriever_metrics.csv", index=False)
rag_quality_df.to_csv("./results/rag_quality.csv", index=False)
quality_df.to_csv("./results/summary.csv", index=False)

print("\n✅ All results saved to ./results/")
print("   - retriever_metrics.csv (MRR, Hit Rate, Precision, Recall)")
print("   - rag_quality.csv (Faithfulness, Relevancy per question)")
print("   - summary.csv (Overall comparison)")



COMPLETE EVALUATION SUMMARY

📊 RETRIEVER METRICS:
   Retriever      MRR  Hit Rate  Precision   Recall
      Vector 0.562879  0.962617   0.019252 0.962617
        BM25 0.495090  0.915888   0.018318 0.915888
Hybrid (RRF) 0.565633  0.953271   0.019065 0.953271

🏆 BEST PERFORMERS:
   MRR: Hybrid (RRF) (0.5656)
   Hit Rate: Vector (0.9626)
   Precision: Vector (0.0193)
   Recall: Vector (0.9626)

📊 RAG QUALITY METRICS:
   Retriever  Avg Faithfulness  Avg Relevancy
      Vector              0.88           0.90
        BM25              0.92           0.94
Hybrid (RRF)              0.92           0.94

✅ All results saved to ./results/
   - retriever_metrics.csv (MRR, Hit Rate, Precision, Recall)
   - rag_quality.csv (Faithfulness, Relevancy per question)
   - summary.csv (Overall comparison)


## 16. Interactive Query Function

In [19]:
def query(question: str):
    """Query the paper using hybrid retrieval."""
    response = hybrid_engine.query(question)
    
    print(f"\n❓ {question}")
    print(f"\n📄 Answer:\n{response.response}")
    print(f"\n📚 Sources ({len(response.source_nodes)}):")
    for i, node in enumerate(response.source_nodes[:3], 1):
        print(f"   {i}. [{node.metadata.get('type', 'text')}] {node.text[:80]}...")
    
    return response

# Test it
query("What is the key innovation of the Transformer?")


❓ What is the key innovation of the Transformer?

📄 Answer:
The Transformer's key innovation is its reliance entirely on attention mechanisms, completely dispensing with recurrent and convolutional neural networks for sequence transduction. It is the first transduction model to compute representations of its input and output solely using self-attention, without employing sequence-aligned RNNs or convolutions.

📚 Sources (10):
   1. [image_description] Figure from page 4: This figure presents a scientific diagram illustrating the a...
   2. [image_description] **5. Key insights or patterns shown:**
*   **Encoder-Decoder Architecture:** The...
   3. [image_description] **2. Number of independent diagrams and their positions:**
This image presents a...


Response(response="The Transformer's key innovation is its reliance entirely on attention mechanisms, completely dispensing with recurrent and convolutional neural networks for sequence transduction. It is the first transduction model to compute representations of its input and output solely using self-attention, without employing sequence-aligned RNNs or convolutions.", source_nodes=[NodeWithScore(node=TextNode(id_='5adfa9cd-a064-4313-94b4-13363447f2ca', embedding=None, metadata={'type': 'image_description', 'content_type': 'figure', 'page': 4}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='3f1b07f0-1ee4-4201-b0bb-c72cad59c3fa', node_type='4', metadata={'type': 'image_description', 'content_type': 'figure', 'page': 4}, hash='9d6647da03abdbe74db9a48700ec49078f4061461b8e022ab4d5b9c887aa8864'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='607fcd4e-7591-4f56-adb5-7206f22c18cb', node_type='1

In [20]:
query("Explain the difference between two diagrams in Figure 2.")


❓ Explain the difference between two diagrams in Figure 2.

📄 Answer:
The two diagrams illustrate distinct but related attention mechanisms. One diagram depicts the Scaled Dot-Product Attention, which is a fundamental attention function. It takes queries (Q), keys (K), and values (V) as input, computes dot products between queries and keys, scales them by the square root of the key dimension, applies a softmax function to obtain weights, and then multiplies these weights by the values to produce an output.

The other diagram illustrates Multi-Head Attention, which is a more complex mechanism built upon Scaled Dot-Product Attention. Instead of performing a single attention function, Multi-Head Attention linearly projects the queries, keys, and values multiple times into different subspaces. It then performs several Scaled Dot-Product Attention operations in parallel, each on a different set of projected queries, keys, and values. The outputs from these parallel attention heads are then

Response(response='The two diagrams illustrate distinct but related attention mechanisms. One diagram depicts the Scaled Dot-Product Attention, which is a fundamental attention function. It takes queries (Q), keys (K), and values (V) as input, computes dot products between queries and keys, scales them by the square root of the key dimension, applies a softmax function to obtain weights, and then multiplies these weights by the values to produce an output.\n\nThe other diagram illustrates Multi-Head Attention, which is a more complex mechanism built upon Scaled Dot-Product Attention. Instead of performing a single attention function, Multi-Head Attention linearly projects the queries, keys, and values multiple times into different subspaces. It then performs several Scaled Dot-Product Attention operations in parallel, each on a different set of projected queries, keys, and values. The outputs from these parallel attention heads are then concatenated and passed through a final linear tr

In [21]:
query("whats the title of the paper?")


❓ whats the title of the paper?

📄 Answer:
The title of the paper is "Attention Is All You Need".

📚 Sources (10):
   1. [metadata] ## Paper Metadata

**Title:** Attention Is All You Need

**Authors:** Ashish Vas...
   2. [text] the input sequence centered around the respective output position. This would in...
   3. [text] For English-French, we used the signiﬁcantly larger WMT 2014 English-French data...


Response(response='The title of the paper is "Attention Is All You Need".', source_nodes=[NodeWithScore(node=TextNode(id_='bf5333a7-bf9c-4e11-945e-6f53515ea9f7', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'type': 'metadata', 'content_type': 'metadata', 'title': 'Attention Is All You Need', 'authors': ['Ashish Vaswani', 'Google Brain', 'Noam Shazeer', 'Niki Parmar', 'Google Research'], 'chunk_id': 0, 'total_chunks': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='## Paper Metadata\n\n**Title:** Attention Is All You Need\n\n**Authors:** Ashish Vaswani, Google Brain, Noam Shazeer, Niki Parmar, Google Research, Jakob Uszkoreit, Llion Jones, Illia Polosukhin\n\n**Contact:** nikip@google.com, llion@google.com, illia.polosukhin@gmail.com, noam@google.com, avaswani@google.com, aidan@cs.toronto.edu, usz@google.com, lukaszkaiser@google.com\n\n**Affiliations:** Goog

In [22]:
# More queries
query("What BLEU score did Transformer achieve on English-German translation?")


❓ What BLEU score did Transformer achieve on English-German translation?

📄 Answer:
The Transformer (big) model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. The base Transformer model achieved a BLEU score of 27.3 on the same task.

📚 Sources (10):
   1. [text] Table 2: The Transformer achieves better BLEU scores than previous state-of-the-...
   2. [text] 6 Results 6.1 Machine Translation On the WMT 2014 English-to-German translation ...
   3. [section] ## 6.1 Machine Translation 27.3 38.1 3.3 · 1018 Transformer (big) 28.4 41.0 2.3 ...


Response(response='The Transformer (big) model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. The base Transformer model achieved a BLEU score of 27.3 on the same task.', source_nodes=[NodeWithScore(node=TextNode(id_='2c3624ea-fd2c-4f90-8386-e2dcfc700eb1', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text', 'content_type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='5e6bd043-d441-4ba3-9389-5362c2e7f77a', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text', 'content_type': 'text'}, hash='44431f3c6643f4d84d2fa2d30f8583b033696f96e6b776a0f3491ed8cf4143ee'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='c43f00bc-1419-490b-a92b-7020876dd6b5', node_type='1', metadata={}, hash='e939fb121c717a5533baf5898e778ae38d25a60549ec451c6ba592575e564bb2')}, metadata_template

In [23]:
query("whats the complexity per layer of self-attention vs recurrent layers?")


❓ whats the complexity per layer of self-attention vs recurrent layers?

📄 Answer:
The complexity per layer for self-attention is O(n^2 · d), while for recurrent layers, it is O(n · d^2).

📚 Sources (10):
   1. [section] As noted in Table 1, a self-attention layer connects all positions with a consta...
   2. [text] We chose the sinusoidal version because it may allow the model to extrapolate to...
   3. [text] Table 1: Maximum path lengths, per-layer complexity and minimum number of sequen...


Response(response='The complexity per layer for self-attention is O(n^2 · d), while for recurrent layers, it is O(n · d^2).', source_nodes=[NodeWithScore(node=TextNode(id_='37879775-2dfd-4a6a-bc6d-ae7a82187913', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'type': 'section', 'content_type': 'section', 'section_name': 'Why Self-Attention', 'section_number': '4'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='83abbd8c-fe48-4558-8fca-64df591583c7', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'type': 'section', 'content_type': 'section', 'section_name': 'Why Self-Attention', 'section_number': '4'}, hash='559f0655dc69e98bb4c9c649eb5df05b8a19df349cb661b9cae213353522d8fd'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='54b25d34-146d-4271-a15a-2845d88815f1', node_type='1', metadata={'source': './attention_is_all_you_need.pdf', 'type': 'secti

In [24]:
query("Describe the encoder architecture from Figure 1.")


❓ Describe the encoder architecture from Figure 1.

📄 Answer:
The encoder component of the model is structured as a stack of six identical layers. It begins by processing the raw input sequence, which first undergoes an Input Embedding to convert tokens into dense vector representations. Positional Encoding is then added to these embeddings to incorporate information about the order of tokens in the sequence.

Each of the six encoder layers consists of two main sub-layers. The first sub-layer is a multi-head self-attention mechanism, which enables each position within the encoder to attend to all positions in the input sequence to compute its representation. The second sub-layer is a simple, position-wise fully connected feed-forward network.

Around each of these two sub-layers, a residual connection is applied, followed by layer normalization. This means the output of each sub-layer is the normalized sum of its input and its processed output. To support these residual connections, a

Response(response='The encoder component of the model is structured as a stack of six identical layers. It begins by processing the raw input sequence, which first undergoes an Input Embedding to convert tokens into dense vector representations. Positional Encoding is then added to these embeddings to incorporate information about the order of tokens in the sequence.\n\nEach of the six encoder layers consists of two main sub-layers. The first sub-layer is a multi-head self-attention mechanism, which enables each position within the encoder to attend to all positions in the input sequence to compute its representation. The second sub-layer is a simple, position-wise fully connected feed-forward network.\n\nAround each of these two sub-layers, a residual connection is applied, followed by layer normalization. This means the output of each sub-layer is the normalized sum of its input and its processed output. To support these residual connections, all sub-layers in the model, as well as t

In [25]:
query("Looking at Table 3, what was the development set BLEU score for the base model after 100,000 steps?")


❓ Looking at Table 3, what was the development set BLEU score for the base model after 100,000 steps?

📄 Answer:
The development set BLEU score for the base model after 100,000 steps was 25.8.

📚 Sources (10):
   1. [text] 6 Results 6.1 Machine Translation On the WMT 2014 English-to-German translation ...
   2. [text] Table 3: Variations on the Transformer architecture. Unlisted values are identic...
   3. [section] While single-head attention is 0.9 BLEU worse than the best setting, quality als...


Response(response='The development set BLEU score for the base model after 100,000 steps was 25.8.', source_nodes=[NodeWithScore(node=TextNode(id_='c43f00bc-1419-490b-a92b-7020876dd6b5', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text', 'content_type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='5e6bd043-d441-4ba3-9389-5362c2e7f77a', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text', 'content_type': 'text'}, hash='44431f3c6643f4d84d2fa2d30f8583b033696f96e6b776a0f3491ed8cf4143ee'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='2c3624ea-fd2c-4f90-8386-e2dcfc700eb1', node_type='1', metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text', 'content_type': 'text'}, hash='2e82fbc162496492f88bf1d926c11d35e4eec45d427438e63b57575024a59040'), <NodeRelationship.NEXT: '

## 17. Cleanup